# Phishing URL Detector Project

## Exploration Data Analysis

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('../data/final_dataset_with_all_features_v3.1.csv')
df.head()

In [ ]:
df['type'].value_counts()

In [ ]:
print(df.shape)
print(df.columns.tolist())

In [ ]:
print(df.isnull().sum())
print(df['label'].value_counts())
print(df.dtypes)

## Modeling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

### Split dataset

In [ ]:
X = df.drop(columns=['url', 'type', 'label', 'scan_date', 'domain'])
y = df['label']

scaler = StandardScaler()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Train Models

#### Logistic Regression Model

In [ ]:
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, solver='saga')
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)

In [ ]:
print(classification_report(y_test, lr_preds, target_names=['benign', 'defacement', 'malware', 'phishing']))

#### Decision Tree Classifier

In [ ]:
dt_model = DecisionTreeClassifier(class_weight='balanced', random_state=42)
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)
print(classification_report(y_test, dt_preds, target_names=['benign', 'defacement', 'malware', 'phishing']))

#### Random Forest

In [ ]:
rf_model = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

print(classification_report(y_test, rf_preds, target_names=['benign', 'defacement', 'malware', 'phishing']))

#### Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance.head(15))

## Visualizations

### Confusion Matrix - Random Forest

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix 

In [ ]:
classes = ['benign', 'defacement', 'malware', 'phishing']
cm = confusion_matrix(y_test, rf_preds)

fig, ax = plt.subplots(figsize=(8,6))

sns.heatmap(cm, 
            annot=True, 
            fmt='d', 
            cmap='Blues',
            xticklabels=classes,
            yticklabels=classes,
            ax=ax)
plt.title("Random Forest Confusion Matrix")

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig('../charts/rf_cm.png')
plt.show()

### Feature Importance

In [ ]:
top_15_features = feature_importance.head(15).sort_values('importance', ascending=True)
features = top_15_features['feature']
importances = top_15_features['importance'].round(3)

In [ ]:
fig, ax = plt.subplots()
bars = ax.barh(features, importances, color='steelblue')
ax.bar_label(bars,
             label_type='center',
             color='white',
             fontweight='bold',
             fontsize=9)
plt.title("Top 15 Feature Importances - Random Forest")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.savefig("../charts/feature_importances.png")
plt.show()

### Model Comparison

In [ ]:
category = ['benign', 'defacement', 'malware', 'phishing']

dt_report = classification_report(y_test, dt_preds, target_names=['benign', 'defacement', 'malware', 'phishing'], output_dict=True)
rf_report = classification_report(y_test, rf_preds, target_names=['benign', 'defacement', 'malware', 'phishing'], output_dict=True)

dt_f1_scores = [dt_report[c]['f1-score'] for c in category]
rf_f1_scores = [rf_report[c]['f1-score'] for c in category]

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
y = np.arange(len(category))
height = 0.33

dt_bars = ax.barh(y - height/2, dt_f1_scores, height, label='Decision Tree F1 Scores', color='steelblue')
rf_bars = ax.barh(y + height/2, rf_f1_scores, height, label='Random Forest F1 Scores', color='darkorange')

ax.set_yticks(y)
ax.set_yticklabels(category)
ax.set_xlim(0, 1.08)
ax.set_xlabel('F1 Score')
ax.legend(loc='upper left', bbox_to_anchor=(1,1))
ax.bar_label(dt_bars, fmt='%.2f', fontsize=8, padding=3)
ax.bar_label(rf_bars, fmt='%.2f', fontsize=8, padding=3)
plt.title("Model Comparison - F1 Score by Class")
plt.savefig("../charts/f1_scores_class.png", bbox_inches='tight')
plt.show()